1. Import & Logging 

In [1]:
import logging
import psycopg2
from psycopg2 import IntegrityError, OperationalError
from psycopg2.extras import RealDictCursor, execute_values

logging.basicConfig(
    filename="etl.log",
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = logging.getLogger("banking_etl")
print("Logging configured > etl.log")



Logging configured > etl.log


2. Connection Wrapper

In [2]:
class DBConnection:
    """ Small wrapper around a psycopg2 connection. """
    def __init__(self, host, port, dbname, user, password):
        self.conn = psycopg2.connect(
            host=host,
            port=port,
            dbname=dbname,
            user=user,
            password=password 
        )

        logger.info("Connected to database '%s'", dbname)

    def cursor(self):
        return self.conn.cursor(cursor_factory=RealDictCursor)

    def commit(self):
        self.conn.commit()

    def rollback(self):
        self.conn.rollback()

    def close(self):
        self.conn.close()
        logger.info("Connection closed")

3. Schema

In [3]:
CREATE_TABLES = """
CREATE TABLE IF NOT EXISTS branches (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    city TEXT NOT NULL,
)


CREATE TABLE IF NOT EXISTS accounts (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    balance NUMERIC(12, 2) NOT NULL DEFAULT 0,
    branch_id INTEGER NOT NULL REFERENCES branches(id),
    created_at TIMESTAMP NOT NULL DEFAULT NOW()

)
"""

def create_schema(db):
    cur = db.cursor()
    cur.execute(CREATE_TABLES)
    db.commit()
    cur.close()
    logger.info("Schema ready")


4. Seed data

In [4]:
BRANCHES = [
    (1, "Kathmandu Main", "Kathmandu"),
    (2, "Lalitpur", "Lalitpur"),
    (3, "Pokhara", "Pokhara")
]

ACCOUNTS = [
    (1, "Alice", 1000.00, 1),
    (2, "Bob", 500.00, 1),
    (3, "Charlie", 750.00, 2),
    (4, "David", 1200.00, 3),
    (5, "Eve", 300.00, 2)
]

def seed(db):
    """ Insert reference data first, then dependent data. Safe to re-run """
    cur = db.cursor()
    try:
        execute_values(cur, """
            INSERT INTO branches (id, name, city) VALUES %s ON CONFLICT (Id) DO NOTHING
        """, BRANCHES)
        logger.info("Seeded %d branches", len(BRANCHES))
        execute_values(cur, """
            INSERT INTO accounts (id, name, balance, branch_id) VALUES %s ON CONFLICT (Id) DO NOTHING
        """, ACCOUNTS)
        logger.info("Seeded %d accounts", len(ACCOUNTS))
        db.commit()

    except IntegrityError as e:
        db.rollback()
        logger.error("Seed failed - integrity: %s", e)
        raise
    finally:
        cur.close()

5. Verify

In [5]:
def verify(db):
    cur = db.cursor()
    cur.execute(""" 
        SELECT b.city, COUNT(*) AS accounts, SUM(a.balance) AS total 
        FROM accounts a 
        JOIN branches b ON b.id = a.branch_id
        GROUP BY b.city
        ORDER BY total DECS
    """)
    for row in cur.fetchall():
        print(f"{row['city']:>12} {row['accounts']} accounts Rs. {row['total']:,}")
        cur.close()


6. Run

In [6]:
db = None
try:
    db = DBConnection(
        host="localhost", port=5432, dbname="banking", user="likhita", password="likhita"
    )
    create_schema(db)
    seed(db)
    verify(db)

except OperationalError as e:
    logger.critical("Cannot reach databases: %s", e)
    print("Connection failed - check host/port/credentials")

except Exception as e:
    if db:
        db.rollback()
    logger.exception("Unexpected error")
    raise
finally:
    if db:
        db.close()

Connection failed - check host/port/credentials


In [7]:
# extract

def extract_transactions(db, since):
    """Pull raw transaction rows. No cleaning here - just read."""
    cur = db.cursor()
    cur.execute(""" 
        SELECT t.id, t.account_id, t.amount, t.txn_type, t.description, t.created_at, a.balance AS current_balance
        FROM transaction t
        JOIN accounts a ON a.id = t.account_id
        WHERE t.created_at >=%s 
        AND t.processed = FALSE
        ORDER BY t.created_at
    """, (since,))
    rows = cur.fetchall()
    cur.close()
    logger.info("Extracted %d raw transactions since %s", len(rows), since)
    return rows


In [8]:
# transform

from datetime import datetime
from decimal import Decimal

def transform(raw_rows):
    """ Pure function: rows in, cleaned rows + rejects out. """
    clean, rejected = [], []

    for row in raw_rows:
        try:
            # ---- validate ----
            if row['amount'] is None:
                raise ValueError("null amount")
            amount = Decimal(str(row['amount']))
            if amount == 0:
                raise ValueError("Zero-value transaction")

            # normalize
            txn_type = (row['txn_type'] or "").strip().upper()
            if txn_type not in {"CREDIT", "DEBIT"}:
                raise ValueError(f"unknown txn_type {txn_type!r}")

            desc = " ".join((row['description'] or "").split())

            # ---- derive ----
            signed = amount if txn_type == "CREDIT" else -amount
            new_balance = Decimal(str(row['current_balance'])) + signed

            if new_balance < 0:
                raise ValueError(f"would overdraw: {new_balance}")

            clean.append({
                "id": row["id"],
                "account_id": row["account_id"],
                "amount": amount,
                "txn_type": txn_type,
                "description": desc,
                "created_at": row["created_at"],
                "new_balance": new_balance,
                "is_large": amount > Decimal("10000.00"),
                "day_of_week": row["created_at"].strftime("%A")
            })

        except (ValueError, TypeError, ArithmeticError) as e:
            rejected.append({"id": row["id"], "reason": str(e)})
            logger.warning("Rejected txn %s: %s", row["id"], e)